## Setup: Imports and Data Loading

In [1]:
import string
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# Kaggle path; fall back to a local 'data' folder if running outside Kaggle
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
import os
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/train.csv'

train_df = pd.read_csv(DATA_PATH)
print(f'Loaded train.csv: {train_df.shape[0]} rows, {train_df.shape[1]} columns')
print(f'Columns: {train_df.columns.tolist()}')
train_df.head(2)

Loaded train.csv: 2000 rows, 8 columns
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


### Helper Functions

Two helpers are used across multiple questions:
- `clean_prompt` lowercases text and strips all characters in `string.punctuation`.
- `map_at_3` computes the MAP@3 score for a single (truth, prediction) pair.


In [2]:
def clean_prompt(text):
    """Lowercase text and remove every character in string.punctuation."""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text


def map_at_3(truth, prediction):
    """
    MAP@3 score for a single question.

    Args:
        truth:      correct answer letter (e.g. 'C')
        prediction: list of up to 3 predicted letters, best guess first

    Returns:
        1.0 / rank if truth is in prediction (rank is 1-indexed), else 0.0
    """
    if truth in prediction:
        rank = prediction.index(truth) + 1
        return 1.0 / rank
    return 0.0


# Quick sanity check on map_at_3 with the two conceptual cases
print('Sanity check - map_at_3:')
print('  truth=C, pred=[C,A,B] ->', map_at_3('C', ['C', 'A', 'B']))  # expected 1.0
print('  truth=B, pred=[D,B,E] ->', map_at_3('B', ['D', 'B', 'E']))  # expected 0.5

Sanity check - map_at_3:
  truth=C, pred=[C,A,B] -> 1.0
  truth=B, pred=[D,B,E] -> 0.5


## Q1. Frequency Distribution of Correct Answers

**Task**: Count occurrences of each answer letter (A, B, C, D, E) in `train.csv`,
then report the sum of the most frequent and least frequent options.

In [3]:
answer_counts = train_df['answer'].value_counts().sort_values(ascending=False)
print('Answer frequency distribution:')
print(answer_counts)
print()

most_freq_letter  = answer_counts.index[0]
least_freq_letter = answer_counts.index[-1]
most_freq_count   = answer_counts.iloc[0]
least_freq_count  = answer_counts.iloc[-1]
total             = most_freq_count + least_freq_count

print(f'Most frequent option : {most_freq_letter} = {most_freq_count}')
print(f'Least frequent option: {least_freq_letter} = {least_freq_count}')
print()
print(f'ANSWER Q1: {most_freq_count} + {least_freq_count} = {total}')

Answer frequency distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent option : B = 490
Least frequent option: E = 324

ANSWER Q1: 490 + 324 = 814


## Q2. Vocabulary Size of the Cleaned Prompt Column

**Task**: Convert the `prompt` column to lowercase, remove every character in
`string.punctuation`, split on whitespace, and count the number of unique words
across the entire column.

In [4]:
# Apply the cleaning function to every prompt
train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)

# Tokenize by whitespace and collect unique tokens
all_prompt_words = []
for text in train_df['prompt_clean']:
    all_prompt_words.extend(text.split())

unique_prompt_words = set(all_prompt_words)
print(f'Total tokens (with repetition): {len(all_prompt_words)}')
print(f'Unique tokens (vocabulary size): {len(unique_prompt_words)}')
print()
print(f'ANSWER Q2: {len(unique_prompt_words)}')

Total tokens (with repetition): 36293
Unique tokens (vocabulary size): 859

ANSWER Q2: 859


## Q3. Stop-Word Filtering on Row ID 1

**Task**: Take the cleaned prompt of the row whose `id == 1`, remove tokens that
appear in `sklearn.feature_extraction.text.ENGLISH_STOP_WORDS`, and count the
remaining tokens.

In [5]:
# Locate Row ID 1 (the row whose 'id' column equals 1)
row_id_1 = train_df[train_df['id'] == 1].iloc[0]
print(f'Original prompt (id=1):')
print(f'  {row_id_1["prompt"]}')
print()

cleaned_prompt_id1 = clean_prompt(row_id_1['prompt'])
print(f'Cleaned prompt (lowercase, no punctuation):')
print(f'  {cleaned_prompt_id1}')
print()

# Split into tokens
tokens_id1 = cleaned_prompt_id1.split()
print(f'Total tokens after cleaning: {len(tokens_id1)}')

# Filter out stop words
filtered_tokens = [w for w in tokens_id1 if w not in ENGLISH_STOP_WORDS]
print(f'Tokens remaining after stop-word filtering: {len(filtered_tokens)}')
print(f'Remaining tokens: {filtered_tokens}')
print()
print(f'ANSWER Q3: {len(filtered_tokens)}')

Original prompt (id=1):
  Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Cleaned prompt (lowercase, no punctuation):
  pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options

Total tokens after cleaning: 22
Tokens remaining after stop-word filtering: 13
Remaining tokens: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']

ANSWER Q3: 13


## Q4. TF-IDF Vocabulary Size on Combined Text

**Task**: For every row in `train.csv`, build a single document by concatenating
the prompt and all five options. Fit a default `TfidfVectorizer(stop_words='english')`
on this list of documents and report the resulting vocabulary size (number of
feature columns).

In [6]:
# Build one document per row: prompt + A + B + C + D + E
def combine_row_text(row):
    parts = [str(row['prompt'])]
    for opt in ['A', 'B', 'C', 'D', 'E']:
        parts.append(str(row[opt]))
    return ' '.join(parts)

combined_documents = train_df.apply(combine_row_text, axis=1).tolist()
print(f'Number of combined documents: {len(combined_documents)}')
print(f'Sample (first 120 chars): {combined_documents[0][:120]}...')
print()

# Fit default TfidfVectorizer with English stop words
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(combined_documents)

vocab_size = len(tfidf_vectorizer.get_feature_names_out())
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size (feature columns): {vocab_size}')
print()
print(f'ANSWER Q4: {vocab_size}')

Number of combined documents: 2000
Sample (first 120 chars): Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? amo...

TF-IDF matrix shape: (2000, 2762)
Vocabulary size (feature columns): 2762

ANSWER Q4: 2762


## Q5. Cosine Similarity Between Prompt and Option A (Row ID 1)

**Task**: Using the TF-IDF vectorizer fitted in Q4, transform the prompt and
option A of Row ID 1 into vectors, and compute their cosine similarity. Round
to four decimal places.

In [7]:
# Reuse the vectorizer fitted in Q4
row1 = train_df[train_df['id'] == 1].iloc[0]
prompt_text = str(row1['prompt'])
option_a_text = str(row1['A'])

print(f'Prompt  (id=1): {prompt_text[:90]}...')
print(f'Option A (id=1): {option_a_text[:90]}...')
print()

# Transform each text into its TF-IDF vector
prompt_vec = tfidf_vectorizer.transform([prompt_text])
option_a_vec = tfidf_vectorizer.transform([option_a_text])

# Cosine similarity - returns a 1x1 matrix
similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]
print(f'Raw cosine similarity: {similarity}')
print()
print(f'ANSWER Q5: {round(similarity, 4)}')


Prompt  (id=1): Pick the best possible answer: What is Martin Heidegger's view on the relationship between...
Option A (id=1): Martin Heidegger believes that humans exist within a time continuum that is infinite and d...

Raw cosine similarity: 0.27202429519891635

ANSWER Q5: 0.272


## Q6. Top-1 Accuracy of TF-IDF Cosine Similarity

**Task**: For every row, compute the cosine similarity between the prompt and
each of the five options. Pick the option with the highest similarity. Report
the percentage of rows where this top-1 prediction matches the correct answer.

In [8]:
# Vectorize all prompts once (reused for every option comparison)
# This avoids re-transforming the same prompt text 5 times per row.
prompt_vectors = tfidf_vectorizer.transform(train_df['prompt'].astype(str).tolist())

correct_matches = 0
total_rows = len(train_df)

for i in range(total_rows):
    row = train_df.iloc[i]
    prompt_vec = prompt_vectors[i]
    # Vectorize all 5 options for this row at once
    option_texts = [str(row[opt]) for opt in 'ABCDE']
    option_vectors = tfidf_vectorizer.transform(option_texts)

    # cosine_similarity(prompt_vec, option_vectors) returns a 1x5 array
    sims = cosine_similarity(prompt_vec, option_vectors)[0]
    predicted_index = int(np.argmax(sims))
    predicted_letter = 'ABCDE'[predicted_index]

    if predicted_letter == row['answer']:
        correct_matches += 1

percentage = correct_matches / total_rows * 100
print(f'Top-1 matches: {correct_matches} / {total_rows}')
print()
print(f'ANSWER Q6: {percentage:.2f}%')

Top-1 matches: 271 / 2000

ANSWER Q6: 13.55%


## Q7. MAP@3 Conceptual - Truth = C, Prediction = C A B

**Task**: Compute the MAP@3 score for a single question where the ground truth
is `C` and the model's top-3 prediction is `C A B` (in that order).

In [9]:
truth_q7 = 'C'
prediction_q7 = ['C', 'A', 'B']

score_q7 = map_at_3(truth_q7, prediction_q7)
print(f'Ground truth: {truth_q7}')
print(f'Prediction  : {prediction_q7}')
print(f'Truth is at rank {prediction_q7.index(truth_q7) + 1} in the prediction')
print()
print(f'ANSWER Q7: {score_q7}')

Ground truth: C
Prediction  : ['C', 'A', 'B']
Truth is at rank 1 in the prediction

ANSWER Q7: 1.0


## Q8. MAP@3 Conceptual - Truth = B, Prediction = D B E

**Task**: Compute the MAP@3 score for a single question where the ground truth
is `B` and the model's top-3 prediction is `D B E` (in that order).

In [10]:
truth_q8 = 'B'
prediction_q8 = ['D', 'B', 'E']

score_q8 = map_at_3(truth_q8, prediction_q8)
print(f'Ground truth: {truth_q8}')
print(f'Prediction  : {prediction_q8}')
print(f'Truth is at rank {prediction_q8.index(truth_q8) + 1} in the prediction')
print()
print(f'ANSWER Q8: {score_q8}')

Ground truth: B
Prediction  : ['D', 'B', 'E']
Truth is at rank 2 in the prediction

ANSWER Q8: 0.5


## Q9. Majority Class Baseline

**Task**: Identify the three most frequent correct answers in `train.csv`. For
every row, predict those three letters in descending frequency order. Compute
the resulting MAP@3 score across the entire training set.

This is the simplest non-trivial baseline. It captures how much of the
leaderboard score can be explained purely by the answer-position prior, with no
use of question content.

In [11]:
# Top-3 most frequent answer letters from Q1
answer_counts = train_df['answer'].value_counts().sort_values(ascending=False)
top3_letters = answer_counts.index[:3].tolist()
top3_counts  = answer_counts.iloc[:3].tolist()
print(f'Top-3 most frequent answers: {top3_letters}')
print(f'Their counts                : {top3_counts}')
print(f'This is the static prediction for every row: {top3_letters}')
print()

# Apply the same static prediction to every row
per_row_scores = train_df['answer'].apply(lambda a: map_at_3(a, top3_letters))
majority_map3 = per_row_scores.mean()

# Breakdown by where the truth lands in the static prediction
for rank, letter in enumerate(top3_letters, start=1):
    n = (train_df['answer'] == letter).sum()
    print(f'  rank {rank} -> {letter}: {n} rows score {1.0/rank:.4f}')
miss_count = total_rows - sum(top3_counts)
print(f'  miss    -> D/E: {miss_count} rows score 0.0000')
print()
print(f'ANSWER Q9: {majority_map3}')

Top-3 most frequent answers: ['B', 'C', 'A']
Their counts                : [490, 459, 369]
This is the static prediction for every row: ['B', 'C', 'A']

  rank 1 -> B: 490 rows score 1.0000
  rank 2 -> C: 459 rows score 0.5000
  rank 3 -> A: 369 rows score 0.3333
  miss    -> D/E: 682 rows score 0.0000

ANSWER Q9: 0.42125


## Q10. TF-IDF Cosine Similarity Pipeline MAP@3

**Task**: For each row, compute the cosine similarity between the prompt and
each of the five options using the Q4 TF-IDF vectorizer. Sort the options by
descending similarity and take the top-3 as the model's prediction. Report
the average MAP@3 across the entire training set.

This is the first content-aware baseline. It uses no learned weights - the only
signal is lexical overlap between the question and each candidate answer.

In [12]:
# Pre-transform all prompts once for efficiency
prompt_vectors = tfidf_vectorizer.transform(train_df['prompt'].astype(str).tolist())

map3_scores = []
total_rows = len(train_df)

for i in range(total_rows):
    row = train_df.iloc[i]
    prompt_vec = prompt_vectors[i]

    # Vectorize all 5 options for this row
    option_texts = [str(row[opt]) for opt in 'ABCDE']
    option_vectors = tfidf_vectorizer.transform(option_texts)

    # Cosine similarity: prompt vs each option (1x5 array)
    sims = cosine_similarity(prompt_vec, option_vectors)[0]

    # Sort option indices by similarity, descending; take top 3
    sorted_indices = np.argsort(sims)[::-1]
    top3_prediction = ['ABCDE'[idx] for idx in sorted_indices[:3]]

    # Score this row
    score = map_at_3(row['answer'], top3_prediction)
    map3_scores.append(score)

tfidf_pipeline_map3 = float(np.mean(map3_scores))

print(f'Rows scored         : {len(map3_scores)}')
print(f'Rows with score 1.0 : {sum(1 for s in map3_scores if s == 1.0)}')
print(f'Rows with score 0.5 : {sum(1 for s in map3_scores if s == 0.5)}')
print(f'Rows with score 0.33: {sum(1 for s in map3_scores if abs(s - 1/3) < 1e-6)}')
print(f'Rows with score 0.0 : {sum(1 for s in map3_scores if s == 0.0)}')
print()
print(f'ANSWER Q10: {tfidf_pipeline_map3}')

Rows scored         : 2000
Rows with score 1.0 : 220
Rows with score 0.5 : 323
Rows with score 0.33: 387
Rows with score 0.0 : 1070

ANSWER Q10: 0.25525


## Final Answer Summary

In [13]:
summary = [
    ('Q1',  'Sum of most + least frequent answer counts',                 f'{most_freq_count + least_freq_count}'),
    ('Q2',  'Vocabulary size of cleaned prompt column',                    f'{len(unique_prompt_words)}'),
    ('Q3',  'Words remaining in Row ID 1 after stop-word filtering',       f'{len(filtered_tokens)}'),
    ('Q4',  'TF-IDF vocabulary size on combined text',                     f'{vocab_size}'),
    ('Q5',  'Cosine sim(prompt, option A) for Row ID 1 (rounded 4dp)',     f'{round(similarity, 4)}'),
    ('Q6',  'Top-1 accuracy of TF-IDF cosine similarity',                  f'{percentage:.2f}%'),
    ('Q7',  'MAP@3 (truth=C, prediction=C A B)',                           f'{score_q7}'),
    ('Q8',  'MAP@3 (truth=B, prediction=D B E)',                           f'{score_q8}'),
    ('Q9',  'Majority Class Baseline MAP@3 on train.csv',                  f'{majority_map3}'),
    ('Q10', 'TF-IDF Pipeline MAP@3 on train.csv',                          f'{tfidf_pipeline_map3}'),
]

print(f'{'Q#':<5} {'Answer':<10}  Description')
print('-' * 80)
for q, desc, ans in summary:
    print(f'{q:<5} {ans:<10}  {desc}')
print()
print('Observations:')
print(f'  - Majority Class baseline ({majority_map3:.4f}) beats the TF-IDF')
print(f'    pipeline ({tfidf_pipeline_map3:.4f}) by a wide margin. This is')
print(f'    expected: the answer-position prior is strong, while lexical')
print(f'    overlap alone rarely identifies the correct option in this')
print(f'    dataset. The TF-IDF pipeline still gives non-trivial signal')
print(f'    above a random baseline (expected MAP@3 ~= 0.10).')

Q#    Answer      Description
--------------------------------------------------------------------------------
Q1    814         Sum of most + least frequent answer counts
Q2    859         Vocabulary size of cleaned prompt column
Q3    13          Words remaining in Row ID 1 after stop-word filtering
Q4    2762        TF-IDF vocabulary size on combined text
Q5    0.272       Cosine sim(prompt, option A) for Row ID 1 (rounded 4dp)
Q6    13.55%      Top-1 accuracy of TF-IDF cosine similarity
Q7    1.0         MAP@3 (truth=C, prediction=C A B)
Q8    0.5         MAP@3 (truth=B, prediction=D B E)
Q9    0.42125     Majority Class Baseline MAP@3 on train.csv
Q10   0.25525     TF-IDF Pipeline MAP@3 on train.csv

Observations:
  - Majority Class baseline (0.4213) beats the TF-IDF
    pipeline (0.2552) by a wide margin. This is
    expected: the answer-position prior is strong, while lexical
    overlap alone rarely identifies the correct option in this
    dataset. The TF-IDF pipeline still gi